# LangChain Complete Guide
## RAG

This notebook covers:
1. RAG Without LangChain (manual implementation)
2. RAG With LangChain (interactive component visualization)


Model used throughout: `gpt-4o-mini`


---
## SECTION 0 - Installation and Setup

In [ ]:
# ---------------------------------------------------------------
# INSTALL DEPENDENCIES
# ---------------------------------------------------------------

!pip install -q openai langchain langchain-openai langchain-community langchain-core langgraph langsmith faiss-cpu tiktoken numpy requests chromadb

print("[SETUP] All dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 

In [ ]:
!pip install -U langchain langchain-community langchain-core pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.2
    Uninstalling langgraph-checkpoint-4.0.2:
      Successfully uninstalled langgraph-checkpoint-4.0.2
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
  Attempting uninstall: lan

In [ ]:
# ---------------------------------------------------------------
# LOAD API KEYS FROM COLAB SECRETS
#
# Add keys in: left sidebar -> key icon (Secrets)
# Toggle "Notebook access" ON for each key.
# ---------------------------------------------------------------

import os
from google.colab import userdata

# --- OpenAI (required) ---
try:
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("Secret is empty.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("[SETUP] OPENAI_API_KEY loaded.")
except Exception as e:
    raise SystemExit(
        "\n[ERROR] Could not load OPENAI_API_KEY from Colab Secrets.\n"
        "  Steps to fix:\n"
        "  1. Click the key icon in the left sidebar\n"
        "  2. Add secret name: OPENAI_API_KEY\n"
        "  3. Paste your OpenAI key as the value\n"
        "  4. Toggle 'Notebook access' ON\n"
        "  5. Re-run this cell\n"
        f"  (Error: {e})"
    )

# --- LangSmith (optional) ---
try:
    LANGCHAIN_API_KEY = userdata.get("LANGCHAIN_API_KEY")
    if LANGCHAIN_API_KEY:
        os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
        os.environ["LANGCHAIN_TRACING_V2"] = "true"
        os.environ["LANGCHAIN_PROJECT"] = "langsmith_demo_pune"
        print("[SETUP] LANGCHAIN_API_KEY loaded. LangSmith tracing enabled.")
    else:
        print("[SETUP] LANGCHAIN_API_KEY not set. LangSmith tracing disabled.")
except Exception:
    print("[SETUP] LANGCHAIN_API_KEY not found in Secrets. LangSmith tracing disabled.")

# --- OpenWeather (optional) ---
try:
    OPENWEATHER_API_KEY = userdata.get("OPENWEATHER_API_KEY")
    if OPENWEATHER_API_KEY:
        os.environ["OPENWEATHER_API_KEY"] = OPENWEATHER_API_KEY
        print("[SETUP] OPENWEATHER_API_KEY loaded. Real weather data enabled.")
    else:
        print("[SETUP] OPENWEATHER_API_KEY not set. Mock weather data will be used.")
except Exception:
    print("[SETUP] OPENWEATHER_API_KEY not found in Secrets. Mock weather data will be used.")

print("\n[SETUP] Environment configured.")

[SETUP] OPENAI_API_KEY loaded.
[SETUP] LANGCHAIN_API_KEY loaded. LangSmith tracing enabled.
[SETUP] OPENWEATHER_API_KEY loaded. Real weather data enabled.

[SETUP] Environment configured.


In [ ]:
# ---------------------------------------------------------------
# INITIALIZE OPENAI CLIENT
# ---------------------------------------------------------------

from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment automatically

print("[SETUP] OpenAI client initialized.")

[SETUP] OpenAI client initialized.


---
## SECTION 1 - RAG Without LangChain

We build every component manually:
- Chunking
- Embedding
- Vector store (in-memory, using cosine similarity)
- Retrieval
- Generation

In [ ]:
# ---------------------------------------------------------------
# COMPONENT 1: Document (raw text we want to query against)
# ---------------------------------------------------------------

RAW_DOCUMENT = """
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.

LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It uses a graph-based architecture where nodes represent computation steps and edges represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.

LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.
It provides tracing capabilities that allow developers to see every step of an LLM chain or agent run.
LangSmith integrates seamlessly with LangChain and LangGraph.

RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved
from a knowledge base and provided to the LLM as context before generating a response.
RAG helps reduce hallucinations and allows the model to answer questions about private or recent data.

Embeddings are numerical vector representations of text. Similar texts produce similar embedding vectors.
This property allows us to find semantically similar documents using distance metrics like cosine similarity.
OpenAI's text-embedding-ada-002 and text-embedding-3-small are commonly used embedding models.

Vector stores are databases optimized for storing and searching embedding vectors.
FAISS (Facebook AI Similarity Search) is a popular open-source vector store for fast nearest-neighbor search.
Other vector stores include Chroma, Pinecone, Weaviate, and Qdrant.

Agents are LLM-powered systems that can use tools to accomplish tasks.
A tool is a function that the agent can call, such as a web search, calculator, or API call.
Agents decide which tools to use and in what order based on the user's query.
"""

print("[RAW DOCUMENT LOADED]")
print(f"Total characters: {len(RAW_DOCUMENT)}")
print(f"Total words: {len(RAW_DOCUMENT.split())}")

[RAW DOCUMENT LOADED]
Total characters: 1948
Total words: 278


In [ ]:
# ---------------------------------------------------------------
# OPTIONAL: Load RAW_DOCUMENT from a PDF file
# Run this cell to upload a PDF and overwrite the default text above.
# Skip this cell if you want to use the default document.
# ---------------------------------------------------------------

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

print("[PDF LOAD] Upload a PDF file...")
uploaded = files.upload()

if uploaded:
    pdf_path = list(uploaded.keys())[0]
    loader = PyPDFLoader(pdf_path)
    pdf_docs = loader.load()
    RAW_DOCUMENT = "\n\n".join(doc.page_content.strip() for doc in pdf_docs if doc.page_content.strip())
    print(f"[PDF LOAD] Loaded {len(pdf_docs)} pages from: {pdf_path}")
    print(f"[PDF LOAD] RAW_DOCUMENT chars: {len(RAW_DOCUMENT)}")
    print(f"[PDF LOAD] RAW_DOCUMENT words: {len(RAW_DOCUMENT.split())}")
    print("\n[PDF LOAD] Preview:\n")
    print(RAW_DOCUMENT[:800])
else:
    print("[PDF LOAD] No file uploaded. Using the default document.")

[PDF LOAD] Upload a PDF file...


[PDF LOAD] No file uploaded. Using the default document.


In [ ]:
# ---------------------------------------------------------------
# COMPONENT 2: Chunking
# Split the document into overlapping chunks
# ---------------------------------------------------------------

def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list:
    """
    Split text into overlapping character-level chunks.
    """
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    chunks = []
    for para in paragraphs:
        start = 0
        while start < len(para):
            end = start + chunk_size
            chunk = para[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start += chunk_size - overlap
    return chunks


chunks = chunk_text(RAW_DOCUMENT, chunk_size=250, overlap=50)

print(f"[CHUNKING] Produced {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (len={len(chunk)}) ---")
    print(chunk)
    print()

[CHUNKING] Produced 14 chunks

--- Chunk 1 (len=228) ---
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.

--- Chunk 2 (len=28) ---
ources, and building agents.

--- Chunk 3 (len=250) ---
LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It uses a graph-based architecture where nodes represent computation steps and edges represent transitions.
LangGraph is particularly

--- Chunk 4 (len=135) ---
s represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and branching logic.

--- Chunk 5 (len=250) ---
LangSmith is a developer platform for debugging, testing, evaluating, and monitoring LLM applications.
It provides tracing capabilities that allow developers to see every step of an LLM chain

In [ ]:
# ---------------------------------------------------------------
# COMPONENT 3: Embeddings
# Batched OpenAI Embeddings
# ---------------------------------------------------------------

import numpy as np

def batch_list(items, batch_size=2000):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def get_embeddings(texts: list, model: str = "text-embedding-3-small") -> list:
    all_vectors = []
    print(f"[EMBEDDINGS] Total texts: {len(texts)}")
    for batch_num, batch in enumerate(batch_list(texts, batch_size=2000), start=1):
        print(f"[EMBEDDINGS] Processing batch {batch_num} (size: {len(batch)})")
        response = client.embeddings.create(input=batch, model=model)
        vectors = [item.embedding for item in response.data]
        all_vectors.extend(vectors)
    print(f"[EMBEDDINGS] Total vectors: {len(all_vectors)}")
    print(f"[EMBEDDINGS] Vector dimension: {len(all_vectors[0])}")
    return all_vectors


chunk_embeddings = get_embeddings(chunks)

print("\n[EMBEDDINGS] Preview of first embedding:")
print(np.round(chunk_embeddings[0][:8], 5))

[EMBEDDINGS] Total texts: 14
[EMBEDDINGS] Processing batch 1 (size: 14)
[EMBEDDINGS] Total vectors: 14
[EMBEDDINGS] Vector dimension: 1536

[EMBEDDINGS] Preview of first embedding:
[ 8.000e-05 -4.890e-03  5.591e-02 -1.295e-02  3.680e-02  7.400e-04
 -6.200e-04  1.648e-02]


In [ ]:
# ---------------------------------------------------------------
# COMPONENT 4: In-Memory Vector Store
# Store chunks + embeddings, support cosine similarity search
# ---------------------------------------------------------------

class SimpleVectorStore:
    def __init__(self):
        self.texts = []
        self.embeddings = []

    def add(self, texts: list, embeddings: list):
        self.texts.extend(texts)
        self.embeddings.extend([np.array(e) for e in embeddings])
        print(f"[VECTOR STORE] Added {len(texts)} documents. Total: {len(self.texts)}")

    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def search(self, query_embedding: list, top_k: int = 3) -> list:
        q = np.array(query_embedding)
        scores = [(self.cosine_similarity(q, e), t) for e, t in zip(self.embeddings, self.texts)]
        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:top_k]


vector_store = SimpleVectorStore()
vector_store.add(chunks, chunk_embeddings)
print(f"[VECTOR STORE] Ready. Holding {len(vector_store.texts)} chunks.")

[VECTOR STORE] Added 14 documents. Total: 14
[VECTOR STORE] Ready. Holding 14 chunks.


In [ ]:
# ---------------------------------------------------------------
# COMPONENT 5: Retrieval + Generation (Full RAG Pipeline)
# ---------------------------------------------------------------

def rag_query_no_langchain(question: str, top_k: int = 3) -> str:
    print(f"\n{'='*60}")
    print(f"[RAG QUERY] Question: {question}")
    print(f"{'='*60}")

    print("\n[STEP 1] Embedding the question...")
    query_embedding = get_embeddings([question])[0]

    print(f"\n[STEP 2] Retrieving top {top_k} chunks from vector store...")
    results = vector_store.search(query_embedding, top_k=top_k)
    for rank, (score, text) in enumerate(results, 1):
        print(f"  Rank {rank} | Similarity: {score:.4f}")
        print(f"  Text: {text[:120]}...")
        print()

    context = "\n\n".join([text for _, text in results])
    prompt = f"""You are a helpful assistant. Use ONLY the context below to answer the question.
If the context does not contain the answer, say "I don't know based on the provided context."

Context:
{context}

Question: {question}

Answer:"""

    print(f"[STEP 3] Prompt built. Context length: {len(context)} chars.")
    print("\n[STEP 4] Calling GPT-4o-mini for answer generation...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a concise and accurate assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content
    print(f"\n[ANSWER]\n{answer}")
    return answer


answer1 = rag_query_no_langchain("What is RAG and why is it useful?")


[RAG QUERY] Question: What is RAG and why is it useful?

[STEP 1] Embedding the question...
[EMBEDDINGS] Total texts: 1
[EMBEDDINGS] Processing batch 1 (size: 1)
[EMBEDDINGS] Total vectors: 1
[EMBEDDINGS] Vector dimension: 1536

[STEP 2] Retrieving top 3 chunks from vector store...
  Rank 1 | Similarity: 0.6537
  Text: RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved
from a knowledge...

  Rank 2 | Similarity: 0.2840
  Text: s represent transitions.
LangGraph is particularly useful for building complex agent workflows that require cycles and b...

  Rank 3 | Similarity: 0.2706
  Text: LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provid...

[STEP 3] Prompt built. Context length: 617 chars.

[STEP 4] Calling GPT-4o-mini for answer generation...

[ANSWER]
RAG stands for Retrieval-Augmented Generation. It is a technique where relevant documents are retrieved from a k

In [ ]:
answer2 = rag_query_no_langchain("What is LangGraph and how does it differ from LangChain?")


[RAG QUERY] Question: What is LangGraph and how does it differ from LangChain?

[STEP 1] Embedding the question...
[EMBEDDINGS] Total texts: 1
[EMBEDDINGS] Processing batch 1 (size: 1)
[EMBEDDINGS] Total vectors: 1
[EMBEDDINGS] Vector dimension: 1536

[STEP 2] Retrieving top 3 chunks from vector store...
  Rank 1 | Similarity: 0.7406
  Text: LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It ...

  Rank 2 | Similarity: 0.6883
  Text: run.
LangSmith integrates seamlessly with LangChain and LangGraph....

  Rank 3 | Similarity: 0.5700
  Text: LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provid...

[STEP 3] Prompt built. Context length: 548 chars.

[STEP 4] Calling GPT-4o-mini for answer generation...

[ANSWER]
LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs, using a graph-based archit

---
## SECTION 2 - RAG With LangChain

Same pipeline but using LangChain components. We log each component explicitly so the internals are visible.

In [ ]:
# ---------------------------------------------------------------
# COMPONENT 1: Document Loading and Chunking with LangChain
# ---------------------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

raw_doc = Document(
    page_content=RAW_DOCUMENT,
    metadata={"source": "manual", "title": "LangChain Overview"}
)

print("[DOCUMENT LOADING]")
print(f"Document metadata: {raw_doc.metadata}")
print(f"Content length: {len(raw_doc.page_content)} chars")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

lc_chunks = splitter.split_documents([raw_doc])

print(f"\n[CHUNKING] Produced {len(lc_chunks)} chunks\n")
for i, chunk in enumerate(lc_chunks, start=1):
    print(f"--- Chunk {i} ---")
    print(f"Length: {len(chunk.page_content)} chars")
    print("Content:")
    print(chunk.page_content)
    print("Metadata:")
    print(chunk.metadata)
    print("-" * 50)
    print()

[DOCUMENT LOADING]
Document metadata: {'source': 'manual', 'title': 'LangChain Overview'}
Content length: 1948 chars

[CHUNKING] Produced 12 chunks

--- Chunk 1 ---
Length: 228 chars
Content:
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).
It provides tools for chaining prompts, managing memory, integrating with external data sources, and building agents.
Metadata:
{'source': 'manual', 'title': 'LangChain Overview'}
--------------------------------------------------

--- Chunk 2 ---
Length: 224 chars
Content:
LangGraph is a library built on top of LangChain that enables building stateful, multi-actor applications with LLMs.
It uses a graph-based architecture where nodes represent computation steps and edges represent transitions.
Metadata:
{'source': 'manual', 'title': 'LangChain Overview'}
--------------------------------------------------

--- Chunk 3 ---
Length: 110 chars
Content:
LangGraph is particularly useful for bui

In [4]:
# ---------------------------------------------------------------
# COMPONENT 2: Embeddings with LangChain
# ---------------------------------------------------------------

from langchain_openai import OpenAIEmbeddings

print("[EMBEDDINGS] Initializing LangChain OpenAIEmbeddings...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

test_text = "What is retrieval augmented generation?"
test_vector = embeddings_model.embed_query(test_text)

print(f"[EMBEDDINGS] Model: text-embedding-3-small")
print(f"[EMBEDDINGS] Embedding dimension: {len(test_vector)}")
print(f"[EMBEDDINGS] Query: '{test_text}'")
print(f"[EMBEDDINGS] First 8 dimensions: {[round(v, 5) for v in test_vector[:8]]}")

ModuleNotFoundError: No module named 'langchain_openai'

In [3]:
# ---------------------------------------------------------------
# COMPONENT 3: Vector Store with ChromaDB
# ---------------------------------------------------------------

from langchain_community.vectorstores import Chroma

print("[VECTOR STORE] Building Chroma vector store from chunks...")

chroma_store = Chroma.from_documents(
    documents=lc_chunks,
    embedding=embeddings_model,
    persist_directory="./chroma_db"
)

print("[VECTOR STORE] Chroma DB created.")
print(f"[VECTOR STORE] Number of indexed documents: {len(lc_chunks)}")

print("\n[VECTOR STORE] Test similarity search for 'vector stores and FAISS':")
test_results = chroma_store.similarity_search_with_score("vector stores and FAISS", k=3)
for rank, (doc, score) in enumerate(test_results, 1):
    print(f"  Rank {rank} | Score: {score:.4f}")
    print(f"  Text: {doc.page_content[:100]}...")
    print()

ModuleNotFoundError: No module named 'langchain_community'

In [2]:
# ---------------------------------------------------------------
# COMPONENT 4: Retriever
# ---------------------------------------------------------------

retriever = chroma_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("[RETRIEVER] Created Chroma retriever.")
print("[RETRIEVER] Search type: similarity | Top-k: 3")

test_query = "How do agents use tools?"
retrieved_docs = retriever.invoke(test_query)
print(f"\n[RETRIEVER] Test query: '{test_query}'")
print(f"[RETRIEVER] Retrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  Doc {i}: {doc.page_content[:120]}...")

NameError: name 'chroma_store' is not defined

In [ ]:
# ---------------------------------------------------------------
# COMPONENT 5: Prompt Template + LLM + RAG Chain
# ---------------------------------------------------------------

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("[LLM] Initializing ChatOpenAI with gpt-4o-mini...")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the question using ONLY the provided context.

If the context does not contain the answer, respond with:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
)

print("[PROMPT] RAG prompt template created.")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("[CHAIN] RAG chain assembled: Retriever -> format_docs -> Prompt -> LLM -> OutputParser")

query = "What is the document about?"
response = rag_chain.invoke(query)
print(f"\n[QUESTION]\n{query}")
print(f"\n[ANSWER]\n{response}")

[LLM] Initializing ChatOpenAI with gpt-4o-mini...
[PROMPT] RAG prompt template created.
[CHAIN] RAG chain assembled: Retriever -> format_docs -> Prompt -> LLM -> OutputParser

[QUESTION]
What is the document about?

[ANSWER]
I don't know based on the provided context.


In [1]:
# ---------------------------------------------------------------
# Run RAG Queries with LangChain
# ---------------------------------------------------------------

def rag_query_langchain(question: str):
    print(f"\n{'='*60}")
    print(f"[RAG QUERY - LangChain] {question}")
    print(f"{'='*60}")
    print("\n[RETRIEVAL LOG] Fetching relevant chunks...")
    retrieved = retriever.invoke(question)
    for i, doc in enumerate(retrieved, 1):
        print(f"  Chunk {i}: {doc.page_content[:120]}...")
    print("\n[LLM] Generating answer...")
    answer = rag_chain.invoke(question)
    print(f"\n[ANSWER]\n{answer}")
    return answer


rag_query_langchain("What is LangSmith used for?")


[RAG QUERY - LangChain] What is LangSmith used for?

[RETRIEVAL LOG] Fetching relevant chunks...


NameError: name 'retriever' is not defined

In [ ]:
rag_query_langchain("What is IIMA")


[RAG QUERY - LangChain] What is IIMA

[RETRIEVAL LOG] Fetching relevant chunks...
  Chunk 1: Business Impact:
 Safer fraud prevention
 AI governance & compliance
 Human oversight for critical actions
 Reduced ...
  Chunk 2:  Network Operations Team
 Billing Team
 Security Team
A Merge Agent consolidates all departmental findings into a fin...
  Chunk 3: Agent Pattern: HITL Governed AI Workflow
Problem Statement
Telecom companies face increasing SIM-swap fraud, OTP abuse, ...

[LLM] Generating answer...

[ANSWER]
I don't know based on the provided context.


"I don't know based on the provided context."